# 06. GraphRAG - Neo4j Text2Cypher 파이프라인

> **왜 GraphRAG가 필요한가요?**
>
> 일반 벡터 RAG는 "비슷한 문서"를 찾는 데 강해요. 하지만 "A 회사의 CEO가 B 회사 이사회에도 참여하는가?" 같은 **관계(relationship) 기반 질문**에는 약해요. GraphRAG는 지식을 그래프로 구조화해서 **엔티티 간 관계를 탐색**할 수 있어요.

> 🔑 **비유**: 벡터 RAG가 **키워드 검색**(비슷한 단어 찾기)이라면, GraphRAG는 **인맥 지도**(누가 누구와 연결되어 있는지 탐색)예요.

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. **Neo4j 그래프 데이터베이스**에 연결하고 Movies 샘플 데이터셋을 로드할 수 있어요
2. **Text2Cypher 파이프라인**을 LangGraph StateGraph로 설계할 수 있어요
3. **자연어 질문을 Cypher 쿼리로 변환**하는 LLM 기반 노드를 구현할 수 있어요
4. **DML 필터링과 조건부 재시도 로직**으로 안전하고 신뢰성 높은 파이프라인을 만들 수 있어요
5. **다양한 그래프 쿼리 패턴**(단순 조회, 집계, 복잡한 관계)을 자연어로 실행할 수 있어요

## 사전 지식

- LangGraph `StateGraph` 기초 (Part 2 - `04-StateGraph-Basics`)
- 조건부 엣지(`add_conditional_edges`) 사용 경험 (Part 2 - `06-Tools-Integration`)
- **벡터 RAG 전체 흐름** (Part 8): 특히 `05-Retrieval.ipynb`에서 **2-Step / Agentic / Multi-Turn** 세 패턴을 비교해요. 이 노트북은 그 비교표의 연장선에서 **그래프 기반 RAG** 라는 네 번째 선택지를 다룹니다
- 이전 노트북: `05-Prompt-Generation.ipynb` (LLM 기반 프롬프트 자동 생성)

> 🧭 **8장 벡터 RAG vs 11/06 GraphRAG — 언제 무엇을 고르나요?**
>
> | 질문 유형 | 적합한 도구 | 이유 |
> |-----------|-----------|------|
> | "양자 얽힘이 뭐야?" 같은 **개념·의미 검색** | Part 8 벡터 RAG | 비정형 텍스트의 의미 유사도가 핵심 |
> | "이 약과 함께 먹으면 안 되는 약은?" 같은 **정의된 관계 탐색** | 이 노트북 (GraphRAG) | 관계가 스키마로 고정된 경우 Cypher가 정확 |
> | **둘 다 필요한 경우** (하이브리드) | 벡터 검색 + Cypher 조합 | `09_Multi_Agent`의 Router/Supervisor로 도구를 나눠 붙여요 |
>
> 요컨대, 8장에서 배운 벡터 RAG가 **"비슷한 문서"** 를 찾는다면, 이 노트북의 GraphRAG는 **"정의된 관계"** 를 질의해요. 둘은 대체재가 아니라 **보완재**예요.

## GraphRAG와 Text2Cypher란?

전통적인 RAG(Retrieval-Augmented Generation)가 벡터 유사도로 문서를 검색한다면, **GraphRAG**는 **그래프 데이터베이스**를 지식 저장소로 사용해요. 그래프는 엔티티 간의 **관계**를 명시적으로 표현하므로, 복잡한 연결 관계를 추론하는 데 강점이 있어요.

**Text2Cypher**는 사용자의 자연어 질문을 Neo4j의 질의 언어인 **Cypher**로 자동 변환하는 기법이에요.

| 항목 | 벡터 RAG | GraphRAG (Text2Cypher) |
|------|----------|------------------------|
| 지식 표현 | 벡터 임베딩 | 노드 + 관계 그래프 |
| 검색 방식 | 코사인 유사도 | Cypher 쿼리 |
| 강점 | 의미 유사성 검색 | 명시적 관계 추론 |
| 적합한 데이터 | 비정형 텍스트 | 구조화된 엔티티 관계 |
| 예시 질문 | "AI에 대해 알려줘" | "Keanu Reeves가 출연한 영화 감독은?" |

> 🔑 **핵심 개념**: GraphRAG는 "누가 누구와 어떤 관계인가?" 같은 **관계형 질문**에 특히 강해요. 지식 그래프(KG)와 LLM을 결합한 형태예요.

### Movies 데이터셋 구조

이 노트북에서는 Neo4j의 Movies 샘플 데이터셋을 사용해요:

| 노드 레이블 | 속성 | 설명 |
|------------|------|------|
| `Movie` | title, released, tagline | 영화 정보 |
| `Person` | name, born | 인물 정보 |

| 관계 타입 | 방향 | 설명 |
|----------|------|------|
| `ACTED_IN` | (Person) → (Movie) | 배우 - 영화 출연 |
| `DIRECTED` | (Person) → (Movie) | 감독 - 영화 감독 |
| `PRODUCED` | (Person) → (Movie) | 프로듀서 - 영화 제작 |

## Text2Cypher 파이프라인 아키텍처

```mermaid
flowchart TD
    START([사용자 질문<br>User Question]) --> GS[get_schema<br>스키마 조회]
    GS --> GC[generate_cypher<br>Cypher 생성]
    GC --> VC[validate_cypher<br>쿼리 검증]
    VC --> EC[execute_cypher<br>쿼리 실행]
    EC --> SR{should_retry<br>결과 판단}
    SR -->|성공| GA[generate_answer<br>자연어 응답]
    SR -->|오류 + 재시도 가능| GC
    SR -->|재시도 초과| GA
    GA --> END([최종 응답<br>Final Answer])

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef decision fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24

    class START input
    class GS,GC,VC,EC process
    class GA,END output
    class SR decision
```

### 파이프라인 구성 노드

| 노드 | 역할 | 입력 | 출력 |
|------|------|------|------|
| `get_schema` | Neo4j 스키마 조회 | - | schema |
| `generate_cypher` | 자연어 → Cypher 변환 (LLM) | question, schema | cypher_query |
| `validate_cypher` | DML 키워드 필터링 | cypher_query | error |
| `execute_cypher` | Neo4j 쿼리 실행 | cypher_query | cypher_result / error |
| `generate_answer` | 결과 → 자연어 응답 (LLM) | cypher_result | answer |

> 🎯 **강의 포인트**: `execute_cypher` 이후 `should_retry` 함수로 분기가 이루어져요. 오류가 있으면 `generate_cypher`로 되돌아가 최대 3번까지 재시도해요. 이 **루프(loop) 패턴**은 실제 에이전트 개발에서 매우 중요해요.

## 1. 환경 설정

In [ ]:
# ---------------------------------------------------
# 환경 설정
# ---------------------------------------------------
# .env 파일에서 API 키를 로드해요
# 필요 키: OPENAI_API_KEY, NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD
from dotenv import load_dotenv

load_dotenv()

In [ ]:
# ---------------------------------------------------
# LangSmith 추적 설정 (선택 사항)
# ---------------------------------------------------
# LangSmith로 파이프라인 실행 과정을 추적할 수 있어요
import os

# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = "LangGraph-GraphRAG-Neo4j"

## 2. Neo4j 데이터베이스 설정

### Neo4j 연결 방법

Neo4j에 연결하는 방법은 두 가지가 있어요.

**방법 1: Neo4j AuraDB (클라우드 - 권장)**
- [Neo4j AuraDB](https://neo4j.com/cloud/platform/aura-graph-database/)에서 무료 인스턴스를 생성해요
- 생성 후 제공되는 연결 정보를 `.env` 파일에 추가해요
- APOC 플러그인이 기본 제공돼요

**방법 2: Docker (로컬 개발)**
```bash
docker run -d --name neo4j \
  -p 7474:7474 -p 7687:7687 \
  -e NEO4J_AUTH=neo4j/your_password \
  -e NEO4J_PLUGINS='["apoc"]' \
  neo4j:latest
```

`.env` 파일에 다음 환경변수를 설정해요:
```
NEO4J_URI=bolt://localhost:7687
NEO4J_USERNAME=neo4j
NEO4J_PASSWORD=your_password
```

> ⚠️ **자주 하는 실수**: Docker로 실행할 때 `NEO4J_PLUGINS='["apoc"]'` 설정을 빠뜨리면 스키마 조회(`refresh_schema()`)가 실패해요. `Neo4jGraph`의 스키마 조회 기능은 내부적으로 APOC 플러그인을 사용하기 때문이에요.

In [ ]:
# ---------------------------------------------------
# Neo4j 연결
# ---------------------------------------------------
# langchain_neo4j 패키지의 Neo4jGraph를 사용해요
# 환경변수에서 연결 정보를 읽어요
import os
from langchain_neo4j import Neo4jGraph

# Neo4j 데이터베이스 연결
graph_db = Neo4jGraph(
    url=os.environ["NEO4J_URI"],
    username=os.environ["NEO4J_USERNAME"],
    password=os.environ["NEO4J_PASSWORD"],
)

# Neo4j 데이터베이스에 연결되었어요.

## 3. Movies 샘플 데이터셋 로드

Neo4j Movies 샘플 데이터셋에는 영화, 배우, 감독 간의 관계 정보가 포함되어 있어요. Cypher 쿼리로 직접 데이터를 생성해서 로드해요.

> 💡 **실무 팁**: 실제 프로덕션에서는 기존 관계형 DB나 CSV 파일에서 Neo4j로 데이터를 마이그레이션하는 경우가 많아요. `LOAD CSV` Cypher 명령어를 사용하면 대량 데이터 적재가 가능해요.

In [ ]:
# ---------------------------------------------------
# Movies 샘플 데이터셋 로드
# ---------------------------------------------------
# Cypher 쿼리로 영화, 인물 노드와 관계를 생성해요
# 포함 영화: Matrix 3부작, Forrest Gump, Cast Away, Top Gun, Jerry Maguire, A Few Good Men
movies_cypher = """
CREATE (TheMatrix:Movie {title:'The Matrix', released:1999, tagline:'Welcome to the Real World'})
CREATE (Keanu:Person {name:'Keanu Reeves', born:1964})
CREATE (Carrie:Person {name:'Carrie-Anne Moss', born:1967})
CREATE (Laurence:Person {name:'Laurence Fishburne', born:1961})
CREATE (Hugo:Person {name:'Hugo Weaving', born:1960})
CREATE (LillyW:Person {name:'Lilly Wachowski', born:1967})
CREATE (LanaW:Person {name:'Lana Wachowski', born:1965})
CREATE (JoelS:Person {name:'Joel Silver', born:1952})

CREATE (TheMatrixReloaded:Movie {title:'The Matrix Reloaded', released:2003, tagline:'Free your mind'})
CREATE (TheMatrixRevolutions:Movie {title:'The Matrix Revolutions', released:2003, tagline:'Everything that has a beginning has an end'})

CREATE (TomH:Person {name:'Tom Hanks', born:1956})
CREATE (ForrestGump:Movie {title:'Forrest Gump', released:1994, tagline:'Life is like a box of chocolates'})
CREATE (RobertZ:Person {name:'Robert Zemeckis', born:1951})
CREATE (CastAway:Movie {title:'Cast Away', released:2000, tagline:'At the edge of the world, his journey begins'})

CREATE (TopGun:Movie {title:'Top Gun', released:1986, tagline:'I feel the need, the need for speed.'})
CREATE (TomC:Person {name:'Tom Cruise', born:1962})
CREATE (TonyS:Person {name:'Tony Scott', born:1944})
CREATE (JerryMaguire:Movie {title:'Jerry Maguire', released:1996, tagline:'The rest of his life begins now.'})
CREATE (CameronC:Person {name:'Cameron Crowe', born:1957})

CREATE (AFewGoodMen:Movie {title:'A Few Good Men', released:1992, tagline:"You can't handle the truth!"})
CREATE (JackN:Person {name:'Jack Nicholson', born:1937})
CREATE (RobR:Person {name:'Rob Reiner', born:1947})

CREATE
  (Keanu)-[:ACTED_IN {roles:['Neo']}]->(TheMatrix),
  (Carrie)-[:ACTED_IN {roles:['Trinity']}]->(TheMatrix),
  (Laurence)-[:ACTED_IN {roles:['Morpheus']}]->(TheMatrix),
  (Hugo)-[:ACTED_IN {roles:['Agent Smith']}]->(TheMatrix),
  (LillyW)-[:DIRECTED]->(TheMatrix),
  (LanaW)-[:DIRECTED]->(TheMatrix),
  (JoelS)-[:PRODUCED]->(TheMatrix),

  (Keanu)-[:ACTED_IN {roles:['Neo']}]->(TheMatrixReloaded),
  (Carrie)-[:ACTED_IN {roles:['Trinity']}]->(TheMatrixReloaded),
  (Laurence)-[:ACTED_IN {roles:['Morpheus']}]->(TheMatrixReloaded),
  (Hugo)-[:ACTED_IN {roles:['Agent Smith']}]->(TheMatrixReloaded),
  (LillyW)-[:DIRECTED]->(TheMatrixReloaded),
  (LanaW)-[:DIRECTED]->(TheMatrixReloaded),

  (Keanu)-[:ACTED_IN {roles:['Neo']}]->(TheMatrixRevolutions),
  (Carrie)-[:ACTED_IN {roles:['Trinity']}]->(TheMatrixRevolutions),
  (Laurence)-[:ACTED_IN {roles:['Morpheus']}]->(TheMatrixRevolutions),
  (Hugo)-[:ACTED_IN {roles:['Agent Smith']}]->(TheMatrixRevolutions),
  (LillyW)-[:DIRECTED]->(TheMatrixRevolutions),
  (LanaW)-[:DIRECTED]->(TheMatrixRevolutions),

  (TomH)-[:ACTED_IN {roles:['Forrest Gump']}]->(ForrestGump),
  (RobertZ)-[:DIRECTED]->(ForrestGump),

  (TomH)-[:ACTED_IN {roles:['Chuck Noland']}]->(CastAway),
  (RobertZ)-[:DIRECTED]->(CastAway),

  (TomC)-[:ACTED_IN {roles:['Maverick']}]->(TopGun),
  (TonyS)-[:DIRECTED]->(TopGun),

  (TomC)-[:ACTED_IN {roles:['Jerry Maguire']}]->(JerryMaguire),
  (CameronC)-[:DIRECTED]->(JerryMaguire),

  (TomC)-[:ACTED_IN {roles:['Lt. Daniel Kaffee']}]->(AFewGoodMen),
  (JackN)-[:ACTED_IN {roles:['Col. Nathan R. Jessep']}]->(AFewGoodMen),
  (RobR)-[:DIRECTED]->(AFewGoodMen)
"""

# 기존 데이터를 모두 삭제하고 새로 로드해요
graph_db.query("MATCH (n) DETACH DELETE n")
graph_db.query(movies_cypher)
# Movies 샘플 데이터셋이 로드되었어요.

In [ ]:
# ---------------------------------------------------
# 데이터셋 확인
# ---------------------------------------------------
# 로드된 데이터의 기본 통계를 확인해요

# 전체 노드 수 조회
node_count = graph_db.query("MATCH (n) RETURN count(n) AS count")[0]["count"]
print(f"전체 노드 수: {node_count}")

# 전체 관계 수 조회
rel_count = graph_db.query("MATCH ()-[r]->() RETURN count(r) AS count")[0]["count"]
print(f"전체 관계 수: {rel_count}")

# 영화 목록 샘플 확인
# 영화 목록 (최신순 5편):
movies = graph_db.query(
    "MATCH (m:Movie) RETURN m.title AS title, m.released AS released ORDER BY m.released DESC LIMIT 5"
)
for m in movies:
    print(f"  - {m['title']} ({m['released']})")

In [ ]:
# ---------------------------------------------------
# 그래프 스키마 확인
# ---------------------------------------------------
# 스키마는 LLM이 올바른 Cypher를 생성하는 데 핵심 정보예요
# 노드 레이블, 관계 유형, 속성 이름이 모두 포함돼요

# 스키마 새로고침
graph_db.refresh_schema()

# 스키마 출력
# === 그래프 스키마 ===
print(graph_db.schema)

## 4. 상태(State) 정의

Text2Cypher 파이프라인에서 각 노드가 공유하는 상태(State)를 정의해요. 각 노드는 이 상태를 읽고 일부 필드를 업데이트해서 다음 노드로 전달해요.

| 필드 | 타입 | 역할 |
|------|------|------|
| `question` | str | 사용자의 자연어 질문 |
| `schema` | str | 그래프 데이터베이스 스키마 |
| `cypher_query` | str | LLM이 생성한 Cypher 쿼리 |
| `cypher_result` | str | 쿼리 실행 결과 |
| `answer` | str | 최종 자연어 응답 |
| `error` | str | 오류 메시지 (오류 없으면 빈 문자열) |
| `retry_count` | int | 현재 재시도 횟수 |

> 🔑 **핵심 개념**: `error`와 `retry_count` 필드가 재시도 로직의 핵심이에요. `should_retry` 함수는 이 두 값을 보고 다음 경로를 결정해요.

In [ ]:
# ---------------------------------------------------
# Text2CypherState 정의
# ---------------------------------------------------
# TypedDict로 상태 필드를 명시적으로 정의해요
# Annotated로 각 필드의 의미를 문서화해요
from typing_extensions import TypedDict, Annotated


class Text2CypherState(TypedDict):
    # 사용자의 자연어 질문
    question: Annotated[str, "User question"]
    # 그래프 데이터베이스 스키마
    schema: Annotated[str, "Graph database schema"]
    # LLM이 생성한 Cypher 쿼리
    cypher_query: Annotated[str, "Generated Cypher query"]
    # Cypher 쿼리 실행 결과
    cypher_result: Annotated[str, "Cypher query result"]
    # 최종 자연어 응답
    answer: Annotated[str, "Final answer"]
    # 오류 메시지 (없으면 빈 문자열)
    error: Annotated[str, "Error message"]
    # 현재 재시도 횟수
    retry_count: Annotated[int, "Retry count"]


print("Text2CypherState 필드:", list(Text2CypherState.__annotations__.keys()))

## 5. LLM 초기화

Cypher 쿼리 생성과 자연어 응답 생성에 사용할 LLM을 초기화해요.

> 💡 **실무 팁**: Text2Cypher에서는 `temperature=0`으로 설정하는 것을 권장해요. 쿼리 생성은 창의성보다 **정확성**이 중요하기 때문이에요. 매번 동일한 질문에 동일한 쿼리가 생성되어야 예측 가능한 동작을 보장할 수 있어요.

In [ ]:
# ---------------------------------------------------
# LLM 초기화
# ---------------------------------------------------
# temperature=0: 쿼리 생성에서 일관성이 중요해요
# 다른 모델 옵션:
#   - "openai:gpt-4o" : 더 복잡한 쿼리에 적합
#   - "anthropic:claude-sonnet-4-5" : Anthropic 모델 사용 시
#   - "ollama:llama3" : 로컬 실행 시
from langchain.chat_models import init_chat_model

llm = init_chat_model("openai:gpt-4o-mini", temperature=0)

# 연결 테스트
test_response = llm.invoke("안녕하세요, 짧게 답해주세요.")
print("LLM 연결 확인:", test_response.content[:50])

## 6. 노드(Node) 구현

Text2Cypher 파이프라인의 5개 노드를 순서대로 구현해요.

### 6-1. get_schema 노드

`get_schema` 노드는 Neo4j 데이터베이스의 최신 스키마를 조회해요. 스키마는 LLM이 올바른 Cypher를 생성하는 데 필수 정보예요.

In [ ]:
# ---------------------------------------------------
# get_schema 노드
# ---------------------------------------------------
# 파이프라인 시작 시 항상 최신 스키마를 조회해요
# refresh_schema(): 캐시를 무효화하고 DB에서 직접 읽어요

def get_schema(state: Text2CypherState) -> dict:
    """Neo4j 그래프 데이터베이스의 스키마를 조회해요."""
    # ==== [GET SCHEMA] ====
    # 스키마 새로고침: 최신 상태 반영
    graph_db.refresh_schema()
    return {"schema": graph_db.schema}

### 6-2. generate_cypher 노드

`generate_cypher` 노드는 LLM을 사용해서 자연어 질문과 스키마를 기반으로 Cypher 쿼리를 생성해요. 시스템 프롬프트에서 DML 사용 금지, 대소문자 처리 방법 등을 명시해요.

In [ ]:
# ---------------------------------------------------
# generate_cypher 노드
# ---------------------------------------------------
# LLM 프롬프트 설계가 핵심이에요
# 스키마 정보를 context로 제공해서 hallucination을 줄여요
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Cypher 생성용 시스템 프롬프트
cypher_generation_system = """You are an expert Neo4j developer. Given a user question and a graph database schema, generate a syntactically correct Cypher query to answer the question.

Instructions:
- Use only the node labels, relationship types, and properties mentioned in the schema.
- Do NOT use any node labels, relationship types, or properties that are not in the schema.
- For string matching, ALWAYS use a WHERE clause with toLower() on BOTH sides:
  CORRECT:   MATCH (p:Person) WHERE toLower(p.name) = toLower('Tom Cruise')
  INCORRECT: MATCH (p:Person {{name: toLower('Tom Cruise')}})
- The property map syntax {{key: value}} performs exact matching - never apply toLower() inside it.
- Always use aliases for nodes and relationships (e.g., `MATCH (m:Movie)` not `MATCH (:Movie)`).
- Return ONLY the Cypher query without any explanation, backticks, or markdown formatting.
- Do NOT include any DML statements (CREATE, DELETE, SET, MERGE, DROP, REMOVE, DETACH, etc.).

Schema:
{schema}"""

# 프롬프트 템플릿 생성
cypher_generation_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", cypher_generation_system),
        ("human", "{question}"),
    ]
)

# Cypher 생성 체인: 프롬프트 → LLM → 문자열 파싱
cypher_chain = cypher_generation_prompt | llm | StrOutputParser()


def generate_cypher(state: Text2CypherState) -> dict:
    """자연어 질문과 스키마를 기반으로 Cypher 쿼리를 생성해요."""
    # ==== [GENERATE CYPHER] ====
    question = state["question"]
    schema = state["schema"]

    # LLM으로 Cypher 생성
    cypher_query = cypher_chain.invoke({"question": question, "schema": schema})

    # 앞뒤 공백 및 마크다운 코드 블록 제거
    cypher_query = cypher_query.strip().strip("`").strip()
    if cypher_query.startswith("cypher"):  # ```cypher 블록 처리
        cypher_query = cypher_query[6:].strip()

    print(f"생성된 Cypher:\n{cypher_query}")
    return {"cypher_query": cypher_query, "error": ""}  # 새 시도이므로 error 초기화

### 6-3. validate_cypher 노드

`validate_cypher` 노드는 생성된 Cypher 쿼리가 **데이터를 변경하는 DML 명령**을 포함하는지 검사해요. 이 안전장치가 없으면 LLM이 실수로 `DELETE`나 `SET` 쿼리를 생성할 수 있어요.

> 🎯 **강의 포인트**: 실제 서비스에서는 읽기 전용 Neo4j 사용자 권한을 부여하는 것이 근본적인 해결책이에요. 하지만 애플리케이션 레벨의 검증도 Defense-in-Depth 원칙에 따라 함께 사용하는 것이 좋아요.

In [ ]:
# ---------------------------------------------------
# validate_cypher 노드
# ---------------------------------------------------
# DML 키워드 차단: 데이터 무결성 보호
# 프로덕션에서는 읽기 전용 DB 계정과 함께 사용해요

# 데이터를 변경하는 Cypher 키워드 목록
DISALLOWED_KEYWORDS = [
    "CREATE",
    "DELETE",
    "SET",
    "MERGE",
    "DROP",
    "REMOVE",
    "DETACH",
]


def validate_cypher(state: Text2CypherState) -> dict:
    """생성된 Cypher 쿼리의 기본 유효성을 검사해요."""
    # ==== [VALIDATE CYPHER] ====
    cypher_query = state["cypher_query"]

    # 빈 쿼리 확인
    if not cypher_query.strip():
        error_msg = "빈 쿼리가 생성되었어요."
        print(f"[VALIDATION FAILED] {error_msg}")
        return {"error": error_msg}

    # DML 키워드 포함 여부 확인 (대소문자 무관)
    query_upper = cypher_query.upper()
    for keyword in DISALLOWED_KEYWORDS:
        if keyword in query_upper:
            error_msg = f"허용되지 않는 키워드가 포함되어 있어요: {keyword}"
            print(f"[VALIDATION FAILED] {error_msg}")
            return {"error": error_msg}

    # [VALIDATION PASSED]
    return {"error": ""}  # 검증 통과

### 6-4. execute_cypher 노드

`execute_cypher` 노드는 검증된 Cypher 쿼리를 Neo4j에서 실행해요. 실행 중 오류가 발생하면 `error` 필드에 기록하고 `retry_count`를 증가시켜요.

In [ ]:
# ---------------------------------------------------
# execute_cypher 노드
# ---------------------------------------------------
# 쿼리 실행 후 결과 또는 오류를 상태에 저장해요
# retry_count는 재시도 횟수를 추적해요

# 최대 재시도 횟수 설정
MAX_RETRIES = 3


def execute_cypher(state: Text2CypherState) -> dict:
    """Cypher 쿼리를 Neo4j 데이터베이스에서 실행해요."""
    # ==== [EXECUTE CYPHER] ====
    cypher_query = state["cypher_query"]
    retry_count = state.get("retry_count", 0)

    # 유효성 검사에서 오류가 있으면 실행하지 않아요
    if state.get("error"):
        return {"retry_count": retry_count + 1}

    try:
        # Neo4j에 쿼리 실행
        result = graph_db.query(cypher_query)
        result_str = str(result)

        # 결과가 비어있는 경우
        if not result:
            result_str = "쿼리가 실행되었지만 결과가 없어요."

        print(f"쿼리 결과 (앞 200자): {result_str[:200]}")
        return {"cypher_result": result_str, "error": ""}

    except Exception as e:
        # 실행 오류: 재시도를 위해 error와 retry_count 업데이트
        error_msg = f"쿼리 실행 오류: {str(e)}"
        print(f"[EXECUTION FAILED] {error_msg}")
        return {"error": error_msg, "retry_count": retry_count + 1}

### 6-5. generate_answer 노드

`generate_answer` 노드는 Cypher 쿼리 결과를 자연어로 요약해서 최종 응답을 생성해요. 오류가 있는 경우에도 사용자 친화적인 메시지를 반환해요.

In [ ]:
# ---------------------------------------------------
# generate_answer 노드
# ---------------------------------------------------
# 쿼리 결과를 자연어로 변환해요
# 오류 상태에서도 사용자에게 친절한 응답을 제공해요

# 응답 생성용 프롬프트
answer_generation_system = """You are a helpful assistant that generates natural language answers based on Neo4j Cypher query results.

Given the user's question, the Cypher query that was executed, and the query result, provide a clear and concise answer in Korean.

If the query result is empty, let the user know politely.
If there was an error, explain what went wrong in simple terms.

Question: {question}
Cypher Query: {cypher_query}
Query Result: {cypher_result}"""

answer_generation_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", answer_generation_system),
        ("human", "위 결과를 바탕으로 질문에 답변해 주세요."),
    ]
)

# 응답 생성 체인
answer_chain = answer_generation_prompt | llm | StrOutputParser()


def generate_answer(state: Text2CypherState) -> dict:
    """Cypher 쿼리 결과를 자연어 응답으로 변환해요."""
    # ==== [GENERATE ANSWER] ====
    question = state["question"]
    cypher_query = state["cypher_query"]
    cypher_result = state.get("cypher_result", "")
    error = state.get("error", "")

    # 오류가 있으면 오류 메시지를 결과로 사용해요
    if error:
        cypher_result = f"오류 발생: {error}"

    # 자연어 응답 생성
    answer = answer_chain.invoke(
        {
            "question": question,
            "cypher_query": cypher_query,
            "cypher_result": cypher_result,
        }
    )

    return {"answer": answer}

## 7. 조건부 엣지 - 재시도 로직

`should_retry` 함수는 `execute_cypher` 노드 이후 다음 경로를 결정해요:

```
오류 없음          → generate_answer (성공)
오류 + 재시도 가능 → generate_cypher  (루프 재시도)
재시도 횟수 초과   → generate_answer (오류 응답)
```

> 🔑 **핵심 개념**: 이 패턴을 **Self-Correction Loop**라고 해요. LLM이 실수한 쿼리를 자동으로 수정해가는 과정이에요. 오류 메시지가 다음 `generate_cypher` 호출에 포함되지는 않지만, LLM이 새로운 시도에서 다른 접근법을 사용할 수 있어요.

In [ ]:
# ---------------------------------------------------
# should_retry 라우팅 함수
# ---------------------------------------------------
# execute_cypher 노드 이후 분기를 결정해요
# 반환 문자열은 add_conditional_edges의 mapping 키와 일치해야 해요
from langgraph.graph import END


def should_retry(state: Text2CypherState) -> str:
    """쿼리 실행 결과에 따라 다음 경로를 결정해요."""
    error = state.get("error", "")
    retry_count = state.get("retry_count", 0)

    if not error:
        # 성공: 바로 응답 생성으로 이동
        print(f"[DECISION: 성공 → generate_answer]")
        return "generate_answer"
    elif retry_count < MAX_RETRIES:
        # 재시도 가능: Cypher 재생성
        print(f"[DECISION: 재시도 ({retry_count}/{MAX_RETRIES}) → generate_cypher]")
        return "generate_cypher"
    else:
        # 재시도 초과: 오류 포함 응답 생성
        print(f"[DECISION: 재시도 초과 ({retry_count}/{MAX_RETRIES}) → generate_answer]")
        return "generate_answer"

## 8. 그래프 구성 및 컴파일

정의한 노드와 엣지를 연결해서 완전한 `StateGraph`를 구성해요.

In [ ]:
# ---------------------------------------------------
# StateGraph 구성
# ---------------------------------------------------
# 노드 추가 → 엣지 연결 → 조건부 엣지 설정 → 컴파일
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

# 1. StateGraph 초기화: Text2CypherState 타입으로 상태 관리
workflow = StateGraph(Text2CypherState)

# 2. 노드 추가
workflow.add_node("get_schema", get_schema)           # 스키마 조회
workflow.add_node("generate_cypher", generate_cypher) # Cypher 생성
workflow.add_node("validate_cypher", validate_cypher) # 쿼리 검증
workflow.add_node("execute_cypher", execute_cypher)   # 쿼리 실행
workflow.add_node("generate_answer", generate_answer) # 응답 생성

# 3. 선형 엣지 연결
workflow.add_edge(START, "get_schema")                 # 시작 → 스키마 조회
workflow.add_edge("get_schema", "generate_cypher")     # 스키마 → Cypher 생성
workflow.add_edge("generate_cypher", "validate_cypher") # Cypher → 검증
workflow.add_edge("validate_cypher", "execute_cypher")  # 검증 → 실행

# 4. 조건부 엣지: execute_cypher 이후 should_retry로 분기
workflow.add_conditional_edges(
    "execute_cypher",
    should_retry,  # 라우팅 함수
    {
        "generate_answer": "generate_answer",  # 성공 또는 초과 → 응답 생성
        "generate_cypher": "generate_cypher",  # 재시도 → Cypher 재생성
    },
)

# 5. 종료 엣지
workflow.add_edge("generate_answer", END)

# 6. 그래프 컴파일: MemorySaver로 체크포인트 활성화
app = workflow.compile(checkpointer=MemorySaver())

# 그래프가 컴파일되었어요.

## 9. 그래프 시각화

구성한 그래프의 노드와 엣지 연결을 시각적으로 확인해요.

In [ ]:
# 그래프 흐름: START → get_schema → generate_cypher → validate_cypher → execute_cypher → (should_retry 분기) → generate_answer → END
# get_schema: Neo4j 그래프 DB의 최신 스키마를 조회해요
# generate_cypher: 자연어 질문과 스키마를 기반으로 Cypher 쿼리를 LLM으로 생성해요
# validate_cypher: DML 키워드(CREATE/DELETE 등)를 차단하여 읽기 전용 동작을 보장해요
# execute_cypher: 검증된 Cypher 쿼리를 Neo4j에서 실행해요
# should_retry: 성공 시 generate_answer로, 오류 + 재시도 가능 시 generate_cypher로 루프해요 (최대 3회)
from IPython.display import Image, display

display(Image(app.get_graph().draw_mermaid_png()))

## 10. 그래프 실행 - 다양한 질문 유형

헬퍼 함수를 정의하고 4가지 유형의 질문으로 파이프라인을 테스트해요.

In [ ]:
# ---------------------------------------------------
# 실행 헬퍼 함수
# ---------------------------------------------------
# 매 실행마다 새로운 thread_id로 독립적인 대화 세션을 시작해요
# 스트리밍으로 각 노드의 실행 결과를 실시간으로 확인해요
import uuid
from langchain_core.runnables import RunnableConfig


def run_text2cypher(question: str) -> dict:
    """Text2Cypher 파이프라인을 실행하는 헬퍼 함수예요."""
    # 매 실행마다 새로운 스레드 ID 생성
    config = RunnableConfig(
        recursion_limit=20,
        configurable={"thread_id": str(uuid.uuid4())},
    )

    # 초기 입력 상태 설정
    inputs = {
        "question": question,
        "retry_count": 0,
    }

    print(f"\n{'='*50}")
    print(f"질문: {question}")
    print(f"{'='*50}")

    # 그래프 스트리밍 실행: 각 노드 업데이트를 실시간으로 출력
    for chunk in app.stream(inputs, config, stream_mode="updates"):
        for node_name, node_output in chunk.items():
            print(f"\n--- [{node_name}] 완료 ---")
            # cypher_query가 있으면 출력
            if "cypher_query" in node_output and node_output["cypher_query"]:
                print(f"  Cypher: {node_output['cypher_query'][:100]}")
            # error가 있으면 출력
            if "error" in node_output and node_output["error"]:
                print(f"  오류: {node_output['error']}")

    # 최종 상태에서 응답 반환
    output = app.get_state(config).values
    return output


# 헬퍼 함수가 준비되었어요.

### 예시 1: 배우 출연작 조회

특정 배우가 출연한 영화 목록을 조회하는 기본 패턴이에요. `ACTED_IN` 관계를 탐색해요.

In [ ]:
# ---------------------------------------------------
# 예시 1: 배우 출연작 조회 (단순 관계 탐색)
# ---------------------------------------------------
output = run_text2cypher("Keanu Reeves가 출연한 영화는 무엇인가요?")

# ==================================================
# 최종 응답:
print(output.get("answer", "응답 없음"))

### 예시 2: 감독 조회

특정 영화의 감독을 조회하는 패턴이에요. `DIRECTED` 관계를 역방향으로 탐색해요.

In [ ]:
# ---------------------------------------------------
# 예시 2: 감독 조회 (역방향 관계 탐색)
# ---------------------------------------------------
output = run_text2cypher("The Matrix를 감독한 사람은 누구인가요?")

# ==================================================
# 최종 응답:
print(output.get("answer", "응답 없음"))

### 예시 3: 집계 쿼리

`COUNT`와 `ORDER BY`를 활용한 집계 쿼리 패턴이에요. 가장 많은 영화에 출연한 배우를 찾아요.

In [ ]:
# ---------------------------------------------------
# 예시 3: 집계 쿼리 (COUNT + ORDER BY)
# ---------------------------------------------------
output = run_text2cypher("가장 많은 영화에 출연한 배우는 누구이며, 몇 편에 출연했나요?")

# ==================================================
# 최종 응답:
print(output.get("answer", "응답 없음"))

### 예시 4: 복잡한 관계 패턴

두 개의 관계를 연결하는 복잡한 패턴이에요. 특정 배우가 출연한 영화의 감독을 찾아요.

> 🎯 **강의 포인트**: 이런 질문이 GraphRAG의 핵심 강점이에요. 벡터 검색으로는 "Tom Hanks와 Robert Zemeckis의 관계"를 명시적으로 표현하기 어렵지만, 그래프 쿼리는 `(Tom Hanks)-[:ACTED_IN]->(Movie)<-[:DIRECTED]-(감독)` 패턴으로 정확히 표현해요.

In [ ]:
# ---------------------------------------------------
# 예시 4: 복잡한 관계 패턴 (다단계 탐색)
# ---------------------------------------------------
output = run_text2cypher("Tom Hanks가 출연한 영화를 감독한 감독은 누구인가요?")

# ==================================================
# 최종 응답:
print(output.get("answer", "응답 없음"))

## 11. 직접 실험해보기

아래 TODO 블록에서 원하는 질문을 입력해서 파이프라인을 실험해보세요.

> 💡 **실무 팁**: 파이프라인이 실패하는 경우를 의도적으로 시도해 보세요. 예를 들어 데이터셋에 없는 배우 이름이나 존재하지 않는 관계를 묻는 질문을 해보면 `empty result` 처리 로직을 확인할 수 있어요.

In [ ]:
# ============================================================
# TODO: 다양한 질문으로 파이프라인을 실험해보세요
# 힌트: 데이터셋에는 Matrix, Forrest Gump, Top Gun, Jerry Maguire,
#       A Few Good Men 등의 영화와 관련 배우/감독 정보가 있어요
# 예상 결과:
#   - 데이터셋에 있는 정보: 정확한 응답
#   - 데이터셋에 없는 정보: "결과가 없다"는 친절한 응답
#   - DML 쿼리 유도 질문: 검증 실패 후 재시도 또는 오류 응답
# ============================================================

# 아래 질문을 수정해서 실험해보세요
my_question = "Tom Cruise가 출연한 영화 목록을 알려주세요."

output = run_text2cypher(my_question)

# ==================================================
# 최종 응답:
print(output.get("answer", "응답 없음"))

## 12. 재시도 로직 심층 탐구

재시도 로직이 실제로 어떻게 동작하는지 살펴볼게요. 스키마에 없는 관계를 질문해서 LLM이 어떻게 오류를 처리하는지 확인해요.

> ⚠️ **자주 하는 실수**: `retry_count`를 초기 입력에 설정하지 않으면 `state.get("retry_count", 0)`에서 기본값 0으로 처리돼요. 하지만 명시적으로 초기화하는 것이 더 안전해요.

In [ ]:
# ---------------------------------------------------
# 재시도 로직 확인
# ---------------------------------------------------
# 결과가 없는 쿼리를 실행해서 empty result 처리를 확인해요
# 실제 재시도는 쿼리 실행 오류(문법 오류 등)에서 발생해요

output = run_text2cypher("데이터베이스에 없는 배우 'Unknown Actor'의 출연작은?")

# ==================================================
# 최종 응답:
print(output.get("answer", "응답 없음"))

# 상태 정보:
print(f"  생성된 Cypher: {output.get('cypher_query', '')}")
print(f"  실행 결과: {output.get('cypher_result', '')}")
print(f"  최종 재시도 횟수: {output.get('retry_count', 0)}")

## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **GraphRAG**: 그래프 데이터베이스를 지식 저장소로 활용하는 RAG 기법으로, 엔티티 간 명시적 관계 추론에 강점을 가져요
- **Text2Cypher 파이프라인**: `get_schema → generate_cypher → validate_cypher → execute_cypher → generate_answer` 5단계로 구성돼요
- **Schema-Aware 프롬프팅**: LLM에게 그래프 스키마를 제공해서 존재하지 않는 레이블/관계를 사용하는 오류를 줄여요
- **DML 필터링**: `CREATE/DELETE/SET/MERGE` 등 데이터 변경 키워드를 차단해서 읽기 전용 동작을 보장해요
- **Self-Correction 루프**: `add_conditional_edges`로 오류 발생 시 자동으로 쿼리를 재생성하는 최대 3회 재시도 메커니즘을 구현해요
- **Neo4jGraph**: `langchain_neo4j`의 `Neo4jGraph` 클래스로 Cypher 실행과 스키마 조회를 통합 관리해요

## 다음 노트북 예고

다음 `08-Data-Analysis-Agent.ipynb`에서는 **데이터 분석 에이전트**를 배워요. 여기서 다룬 Text2Cypher는 그래프 DB의 관계를 질의하는 사례였고, 다음 노트북은 CSV·표 형식 데이터를 읽고 분석하며 시각화와 인사이트 생성까지 자동화하는 사례예요. 그래프 DB의 관계형 지식에서 표 형식 데이터 분석으로 응용 범위를 넓혀요.

<!-- AUTOPILOT_CREATE_AGENT_DEEP_AGENT_APPENDIX -->
## 보강: `create_agent`로 GraphRAG/Text2Cypher를 도구 기반 에이전트로 만들기

### 참고 공식 문서
- [Retrieval / Agentic RAG](https://docs.langchain.com/oss/python/langchain/retrieval)
- [LangChain Agents](https://docs.langchain.com/oss/python/langchain/agents)

본문은 `get_schema → generate_cypher → validate_cypher → execute_cypher → generate_answer`를 `StateGraph`로 직접 연결합니다. 이 방식은 Text2Cypher 파이프라인의 각 안전장치와 재시도 조건을 명시적으로 보여주는 데 좋습니다.

공식 retrieval 문서는 Agentic RAG를 “에이전트가 필요할 때 검색 도구를 호출하는 구조”로 설명합니다. GraphRAG도 같은 관점으로 볼 수 있어요. Neo4j 스키마 조회와 읽기 전용 Cypher 실행을 도구로 제공하면, `create_agent`가 다음을 스스로 결정합니다.

- 언제 스키마를 다시 확인할지
- 어떤 관계/노드가 질문과 관련 있는지
- 어떤 Cypher를 작성하고 실행할지
- 결과가 비었거나 오류가 났을 때 어떻게 수정할지

다만 Graph DB는 쓰기 쿼리 위험이 크기 때문에, **도구 레벨에서 읽기 전용 검증을 반드시 유지**해야 합니다. 즉, `create_agent`로 단순화하더라도 보안 경계는 프롬프트가 아니라 코드에 둬야 해요.


In [ ]:
# ============================================================
# 선택 실행: create_agent 기반 GraphRAG/Text2Cypher Agent
# ============================================================
RUN_CREATE_AGENT_GRAPHRAG_APPENDIX = False

if RUN_CREATE_AGENT_GRAPHRAG_APPENDIX:
    import re
    from typing import List

    from langchain.agents import create_agent
    from langchain.chat_models import init_chat_model
    from langchain.tools import tool
    from pydantic import BaseModel, Field

    # --------------------------------------------------------
    # 1) 구조화된 최종 답변 스키마
    # --------------------------------------------------------
    class GraphRAGAnswer(BaseModel):
        answer: str = Field(description="사용자에게 보여줄 한국어 답변")
        cypher_query: str = Field(description="최종 실행한 읽기 전용 Cypher")
        evidence: List[str] = Field(description="결과 행 또는 근거 요약")
        limitations: str = Field(description="결과 해석 시 주의할 점")

    # --------------------------------------------------------
    # 2) 도구 레벨 읽기 전용 안전장치
    # --------------------------------------------------------
    WRITE_KEYWORDS = re.compile(r"\b(CREATE|MERGE|SET|DELETE|DETACH|DROP|REMOVE|ALTER|LOAD\s+CSV)\b", re.I)

    @tool(parse_docstring=True)
    def get_graph_schema() -> str:
        """Return the current Neo4j graph schema."""
        # 본문에서 만든 graph_db 객체를 재사용해요.
        graph_db.refresh_schema()
        return graph_db.schema

    @tool(parse_docstring=True)
    def run_read_only_cypher(cypher: str) -> str:
        """Run a read-only Cypher query against Neo4j.

        Args:
            cypher: Cypher query that must only read data.
        """
        # 프롬프트만 믿지 않고 코드에서 DML/DDL을 차단해요.
        if WRITE_KEYWORDS.search(cypher):
            return "Error: Only read-only Cypher queries are allowed."
        try:
            rows = graph_db.query(cypher)
            return str(rows[:20]) if rows else "No rows returned."
        except Exception as exc:
            # 오류 문자열을 모델에게 돌려주면 create_agent 루프가 쿼리를 고쳐볼 수 있어요.
            return f"Error: {exc}"

    # --------------------------------------------------------
    # 3) create_agent GraphRAG 에이전트
    # --------------------------------------------------------
    graphrag_agent = create_agent(
        model=init_chat_model("openai:gpt-4o-mini", temperature=0),
        tools=[get_graph_schema, run_read_only_cypher],
        response_format=GraphRAGAnswer,
        system_prompt="""
너는 Neo4j GraphRAG/Text2Cypher 전문가다.
항상 먼저 get_graph_schema로 최신 스키마를 확인한다.
그 다음 사용자의 자연어 질문을 읽기 전용 Cypher로 변환하고 run_read_only_cypher로 실행한다.
쿼리 오류가 반환되면 스키마와 오류 메시지를 바탕으로 한 번 이상 수정한다.
데이터 변경 쿼리는 절대 작성하지 않는다.
최종 답변에는 answer, cypher_query, evidence, limitations를 구조화해서 제공한다.
""",
    )

    result = graphrag_agent.invoke({
        "messages": [{"role": "user", "content": "Tom Hanks가 출연한 영화의 감독을 알려줘."}]
    })

    import json
    print(json.dumps(result["structured_response"].model_dump(), indent=2, ensure_ascii=False))
